In [ ]:
#Verify GPU is on
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu128
GPU available: True
GPU name: Tesla T4


In [ ]:
#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#create project folder
import os
project_dir = '/content/drive/MyDrive/ewallet-sentiment-analysis'
os.makedirs(project_dir, exist_ok=True)
print(f"✅ Project folder ready: {project_dir}")

✅ Project folder ready: /content/drive/MyDrive/ewallet-sentiment-analysis


In [ ]:
#Upload via Colab
from google.colab import files
uploaded = files.upload()   # opens file picker

# Move uploaded file to the project folder
import shutil
for filename in uploaded.keys():
    shutil.move(filename, f"{project_dir}/{filename}")
print("✅ Uploaded")

Saving reviews_cleaned.csv to reviews_cleaned.csv
✅ Uploaded


In [ ]:
#Load the data reviews_cleaned.csv in Colab
import pandas as pd

df = pd.read_csv(f"{project_dir}/reviews_cleaned.csv")
print(f"Loaded {len(df):,} reviews")
print(df["lang_group"].value_counts())
print(df["app_name"].value_counts())

Loaded 137,182 reviews
lang_group
english    77744
malay      46789
other      12649
Name: count, dtype: int64
app_name
Touch n Go eWallet    59361
Boost                 32036
MAE by Maybank        25947
Setel                 17294
GXBank                 2544
Name: count, dtype: int64


In [ ]:
#Phase 4
#Install + load the transformer model
%pip install -q transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

# Label mapping (the model outputs indices 0/1/2)
LABEL_MAP = {0: "negative", 1: "neutral", 2: "positive"}

print(f"✅ Model loaded on {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model loaded on cuda


In [ ]:
#Run inference with batching + checkpoints
import numpy as np
from tqdm import tqdm
import os

CHECKPOINT_PATH = f"{project_dir}/transformer_results_checkpoint.csv"
BATCH_SIZE = 64  # for GPU inference
CHUNK_SIZE = 5000  # save every 5000 rows

def predict_sentiment_batch(texts):
    """Run model on a list of texts, return scores + labels."""
    inputs = tokenizer(
        texts, return_tensors="pt", padding=True,
        truncation=True, max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()

    pred_indices = probs.argmax(axis=1)
    pred_labels = [LABEL_MAP[i] for i in pred_indices]
    # Continuous score: positive_prob - negative_prob (analog to VADER compound)
    pred_scores = probs[:, 2] - probs[:, 0]
    return pred_labels, pred_scores, probs

# Resume from checkpoint if it exists
if os.path.exists(CHECKPOINT_PATH):
    checkpoint = pd.read_csv(CHECKPOINT_PATH)
    start_idx = len(checkpoint)
    print(f"⏯️ Resuming from row {start_idx:,}")
    all_labels = checkpoint["transformer_label"].tolist()
    all_scores = checkpoint["transformer_score"].tolist()
    all_neg = checkpoint["prob_negative"].tolist()
    all_neu = checkpoint["prob_neutral"].tolist()
    all_pos = checkpoint["prob_positive"].tolist()
else:
    start_idx = 0
    all_labels, all_scores = [], []
    all_neg, all_neu, all_pos = [], [], []

texts = df["content"].astype(str).tolist()
total = len(texts)

with tqdm(total=total, initial=start_idx) as pbar:
    for chunk_start in range(start_idx, total, CHUNK_SIZE):
        chunk_end = min(chunk_start + CHUNK_SIZE, total)
        chunk_texts = texts[chunk_start:chunk_end]

        # Process chunk in batches
        for batch_start in range(0, len(chunk_texts), BATCH_SIZE):
            batch_texts = chunk_texts[batch_start:batch_start + BATCH_SIZE]
            labels, scores, probs = predict_sentiment_batch(batch_texts)
            all_labels.extend(labels)
            all_scores.extend(scores.tolist())
            all_neg.extend(probs[:, 0].tolist())
            all_neu.extend(probs[:, 1].tolist())
            all_pos.extend(probs[:, 2].tolist())
            pbar.update(len(batch_texts))

        # Save checkpoint every chunk
        checkpoint_df = pd.DataFrame({
            "transformer_label": all_labels,
            "transformer_score": all_scores,
            "prob_negative": all_neg,
            "prob_neutral": all_neu,
            "prob_positive": all_pos,
        })
        checkpoint_df.to_csv(CHECKPOINT_PATH, index=False)

print("✅ Inference complete")

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

⏯️ Resuming from row 137,182



100%|██████████| 137182/137182 [00:00<?, ?it/s]

✅ Inference complete


In [ ]:
#Merge Results back into Dataframe
results = pd.read_csv(CHECKPOINT_PATH)
df = pd.concat([df.reset_index(drop=True), results], axis=1)

print(df[["app_name", "lang_group", "score", "content",
          "transformer_label", "transformer_score"]].head())

# Save final dataset
df.to_csv(f"{project_dir}/reviews_with_transformer.csv", index=False)
print(f"✅ Saved {len(df):,} reviews with transformer sentiment")

             app_name lang_group  score  \
0  Touch n Go eWallet      malay      5   
1  Touch n Go eWallet    english      5   
2  Touch n Go eWallet    english      2   
3  Touch n Go eWallet    english      5   
4  Touch n Go eWallet    english      1   

                                             content transformer_label  \
0                               apps yg sngt berguna          positive   
1                     Easy to Make payment all bills           neutral   
2  always lock me out whenever i receive funds fr...          negative   
3        Good service. Huge function Love to use TNG          positive   
4  after update today 3 june 2026 app keeps crash...          negative   

   transformer_score  
0           0.439778  
1           0.229538  
2          -0.946165  
3           0.883397  
4          -0.929181  
✅ Saved 137,182 reviews with transformer sentiment


In [ ]:
#The big reveal: did the transformer beat VADER?
# Cross-tab: star rating vs transformer label (compare to VADER's 51% for 1-star)
crosstab = pd.crosstab(
    df["score"],
    df["transformer_label"],
    normalize="index"
) * 100

print("Star rating vs Transformer label (% within each rating):")
print(crosstab.round(1))

Star rating vs Transformer label (% within each rating):
transformer_label  negative  neutral  positive
score                                         
1                      91.6      6.8       1.6
2                      83.7     13.6       2.7
3                      68.6     21.3      10.1
4                      34.5     25.5      40.0
5                      11.5     18.6      69.9


In [ ]:
#Sentiment per app, all languages
# Average transformer score per app — now includes Malay reviews!
avg_sentiment = (
    df.groupby("app_name")["transformer_score"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)
print(avg_sentiment.round(3))

                     mean  median  count
app_name                                
Setel               0.246   0.427  17294
Boost              -0.213  -0.453  32036
Touch n Go eWallet -0.226  -0.471  59361
GXBank             -0.273  -0.601   2544
MAE by Maybank     -0.467  -0.733  25947


In [ ]:
from google.colab import files
files.download(f"{project_dir}/reviews_with_transformer.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =============================================================
# Validate transformer quality against star ratings (ground truth)
#
# Note: VADER vs Transformer comparison is done on the local laptop,
# where both result files exist. This Colab notebook is for
# transformer inference only.
# =============================================================

print("=== TRANSFORMER VALIDATION AGAINST STAR RATINGS ===\n")

# How well does the transformer agree with star ratings?
one_star = df[df["score"] == 1]
five_star = df[df["score"] == 5]

print(f"1-star reviews labeled 'negative': "
      f"{(one_star['transformer_label'] == 'negative').mean() * 100:.1f}%")

print(f"5-star reviews labeled 'positive': "
      f"{(five_star['transformer_label'] == 'positive').mean() * 100:.1f}%")

# Full breakdown
print("\n=== STAR RATING vs TRANSFORMER LABEL (% within each rating) ===")
crosstab = pd.crosstab(
    df["score"],
    df["transformer_label"],
    normalize="index"
) * 100
print(crosstab.round(1))

# Per-language quality check
print("\n=== TRANSFORMER ACCURACY BY LANGUAGE ===")
print("(Higher = better. 1-star → negative is the key metric.)\n")

for lang in df["lang_group"].unique():
    sub = df[df["lang_group"] == lang]
    one_star_sub = sub[sub["score"] == 1]
    if len(one_star_sub) > 0:
        pct = (one_star_sub["transformer_label"] == "negative").mean() * 100
        print(f"  {lang:10s}: {pct:.1f}% of 1-star reviews labeled negative "
              f"(n={len(one_star_sub):,})")

=== TRANSFORMER VALIDATION AGAINST STAR RATINGS ===

1-star reviews labeled 'negative': 91.6%
5-star reviews labeled 'positive': 69.9%

=== STAR RATING vs TRANSFORMER LABEL (% within each rating) ===
transformer_label  negative  neutral  positive
score                                         
1                      91.6      6.8       1.6
2                      83.7     13.6       2.7
3                      68.6     21.3      10.1
4                      34.5     25.5      40.0
5                      11.5     18.6      69.9

=== TRANSFORMER ACCURACY BY LANGUAGE ===
(Higher = better. 1-star → negative is the key metric.)

  malay     : 91.0% of 1-star reviews labeled negative (n=18,709)
  english   : 92.8% of 1-star reviews labeled negative (n=34,974)
  other     : 84.0% of 1-star reviews labeled negative (n=3,417)


In [ ]:
malay_sample = df[df["lang_group"] == "malay"].sample(8, random_state=42)
for _, row in malay_sample.iterrows():
    print(f"[⭐{row['score']}] [{row['transformer_label']:8s}] (score={row['transformer_score']:+.2f}) {row['content'][:150]}")

[⭐5] [positive] (score=+0.69) Dah okey.terima kasih.
[⭐1] [negative] (score=-0.51) asyik connection time out bila nk login. dah bertahun dah xde perubahan
[⭐3] [negative] (score=-0.34) tak boleh nak split screen.. kita nie byk kerja kadang2 satu lagi phone or tablet ada bnde lain nak buat.. sekali x ley split utk buat dua kerja dlm 1
[⭐5] [negative] (score=-0.78) kenapa saya tidak dapat bantuan kerajaan etunai rm100
[⭐5] [positive] (score=+0.05) Saya harap..syg telah bagi 5star dapat la buka utk pembelian krdt tu..dah bnyk kali pasang nyahpasang pasang nyahpasang touch&go ni..tapi tetap juga t
[⭐5] [positive] (score=+0.30) Memudahkan..tak perlu beratur lagi menunggu..dapat point lagi..tq setel.
[⭐1] [negative] (score=-0.88) Careline entah apa apa, tak membantu pon
[⭐5] [neutral ] (score=-0.02) Mcm mna nak buat account baru?
